In [3]:
%pip install boto3

  Using cached boto3-1.42.63-py3-none-any.whl.metadata (6.7 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.16.0-py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 29.3 MB/s  0:00:00 eta 0:00:01
Using cached jmespath-1.1.0-py3-none-any.whl (20 kB)
Using cached s3transfer-0.16.0-py3-none-any.whl (86 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [boto3]32m2/4 [s3transfer]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import boto3
from botocore.client import Config
import pandas as pd
from io import BytesIO
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
USER = os.getenv("MINIO_USER")   
PASSWORD = os.getenv("MINIO_PASSWORD")   

# 1. Connect to MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='miniopassword',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

bucket_name = "market-data"

# 2. Create bucket if it doesn't exist
if bucket_name not in [b['Name'] for b in s3.list_buckets()['Buckets']]:
    s3.create_bucket(Bucket=bucket_name)

# 3. SAVE Data: Upload a Pandas DataFrame as Parquet
df = pd.DataFrame({'ticker': ['AAPL', 'TSLA'], 'price': [150.25, 240.50]})
parquet_buffer = BytesIO()
df.to_parquet(parquet_buffer, index=False)
skey='ticks/2026-03-08_trades.parquet'

s3.put_object(
    Bucket=bucket_name, 
    Key=skey, 
    Body=parquet_buffer.getvalue()
)
print("Data saved to MinIO.")

# 4. READ Data: Pull Parquet back into Pandas
response = s3.get_object(Bucket=bucket_name, Key=skey)
df_read = pd.read_parquet(BytesIO(response['Body'].read()))

print(df_read)


Data saved to MinIO.
  ticker   price
0   AAPL  150.25
1   TSLA  240.50
